# 04 · Collect Docker FAQs, forum solutions and (optionally) Reddit

Three smaller sources, each written as its own file under `raw_sources/`:

| Part | Source | Type label | Notes |
|---|---|---|---|
| A | Docker docs FAQ + troubleshooting pages (raw Markdown from `docker/docs`) | `official_docs_entry` | Highest trust. **These duplicate your KB**, so notebook 05 tags them (`overlaps_kb`) and you decide whether to keep them as tickets. Useful as ground truth for evaluation. |
| B | Docker Community Forums (Discourse) — topics with an accepted solution | `forum_solved` | ~12% of topics are solved (~4k). Polite 1 request/second; the `/search` path that `robots.txt` disallows is never used. Forum terms are not published, so check them before redistributing. |
| C | Reddit (optional, **off by default**) | `reddit_top_comment` | Lowest trust: no accepted-answer marker. Uses Reddit's official API through your own free app credentials, not HTML scraping. Reddit's API terms restrict commercial and AI-training use, so read them first. |

Parts A and B need no keys. Part C needs `REDDIT_CLIENT_ID` / `REDDIT_CLIENT_SECRET` as Colab secrets.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
FORUM_CATEGORIES = {          # slug -> Discourse category id
    "docker-desktop": 48, "docker-engine": 22, "docker-hub": 10,
    "general-discussions": 23, "image-builds": 105, "support": 106,
}
FORUM_MAX_TOPICS = 4000        # solved topics to keep in total
FORUM_MAX_INDEX_PAGES = 600    # listing pages (30 topics each) per category
FORUM_MIN_POSTS, FORUM_MAX_POSTS = 2, 60
REQUEST_DELAY = 1.0            # seconds between forum requests

ENABLE_REDDIT = False          # read the notes above first
REDDIT_SUBREDDITS = ["docker"]
REDDIT_MAX_POSTS = 3000
USE_DRIVE = True

!pip install -q requests

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = Path("/content/drive/MyDrive/customer-support-triage/tickets")
    except ImportError:
        OUT_ROOT = Path("tickets")
else:
    OUT_ROOT = Path("tickets")

RAW_SRC = OUT_ROOT / "raw_sources"
for sub in ("docs_faq", "forum", "reddit"):
    (RAW_SRC / sub).mkdir(parents=True, exist_ok=True)


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")


print("Writing to:", OUT_ROOT.resolve())

## Part A · Docker docs FAQ and troubleshooting entries

Each FAQ question (an H3 heading, usually ending in `?`) becomes a problem, with the text under it as the answer.
On troubleshooting pages, an `#### Error message` / `#### Symptoms` sub-section is moved into the problem and the rest
(possible causes, solution) becomes the resolution.

In [ ]:
DOCS_RAW = "https://raw.githubusercontent.com/docker/docs/main/content/manuals/"
DOCS_SITE = "https://docs.docker.com/"
DOCS_LICENSE = "Apache-2.0 (docker/docs)"

DOC_PAGES = [   # (path under content/manuals/, page kind)
    ("desktop/troubleshoot-and-support/faqs/general.md", "faq"),
    ("desktop/troubleshoot-and-support/faqs/linuxfaqs.md", "faq"),
    ("desktop/troubleshoot-and-support/faqs/macfaqs.md", "faq"),
    ("desktop/troubleshoot-and-support/faqs/windowsfaqs.md", "faq"),
    ("desktop/troubleshoot-and-support/faqs/releases.md", "faq"),
    ("faqs/accounts.md", "faq"),
    ("faqs/platform.md", "faq"),
    ("faqs/security.md", "faq"),
    ("faqs/subscription-billing.md", "faq"),
    ("compose/support-and-feedback/faq.md", "faq"),
    ("enterprise/enterprise-deployment/faq.md", "faq"),
    ("enterprise/security/hardened-desktop/enhanced-container-isolation/faq.md", "faq"),
    ("desktop/troubleshoot-and-support/troubleshoot/topics.md", "troubleshoot"),
    ("engine/daemon/troubleshoot.md", "troubleshoot"),
    ("engine/security/rootless/troubleshoot.md", "troubleshoot"),
    ("docker-hub/troubleshoot.md", "troubleshoot"),
    ("docker-hub/repos/manage/builds/troubleshoot.md", "troubleshoot"),
    ("security/authentication/single-sign-on/troubleshoot-sso.md", "troubleshoot"),
    ("security/provisioning/troubleshoot-provisioning.md", "troubleshoot"),
]

HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")
FRONT_MATTER_RE = re.compile(r"\A---\n.*?\n---\n", re.S)
SHORTCODE_RE = re.compile(r"\{\{[<%].*?[>%]\}\}", re.S)
IMAGE_RE = re.compile(r"!\[[^\]]*\]\([^)]*\)")
PROBLEM_PART_RE = re.compile(r"(?i)^(error( message)?s?|symptoms?|problem|issue)$")


def clean_doc_markdown(text):
    text = FRONT_MATTER_RE.sub("", text)
    text = SHORTCODE_RE.sub("", text)
    text = IMAGE_RE.sub("", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def parse_sections(md):
    # -> list of (level, title, body_lines), ignoring '#' lines inside code fences
    sections, in_fence = [], False
    for line in md.splitlines():
        if line.startswith("```"):
            in_fence = not in_fence
        m = None if in_fence else HEADING_RE.match(line)
        if m:
            sections.append((len(m.group(1)), m.group(2), []))
        elif sections:
            sections[-1][2].append(line)
    return sections


def entry_level(sections, kind):
    counts = {}
    for level, title, _ in sections:
        if level >= 2 and (kind == "troubleshoot" or title.endswith("?")):
            counts[level] = counts.get(level, 0) + 1
    if kind == "troubleshoot":
        return 3 if counts.get(3, 0) >= 2 else 2
    return max(counts, key=counts.get) if counts else 3


def split_problem_and_resolution(subsections):
    problem_parts, resolution_parts = [], []
    for title, body in subsections:
        text = body.strip()
        if not text:
            continue
        if title is None:
            resolution_parts.append(text)
        elif PROBLEM_PART_RE.match(title):
            problem_parts.append(text)
        else:
            resolution_parts.append(f"**{title}**\n{text}")
    return "\n\n".join(problem_parts), "\n\n".join(resolution_parts)


def doc_entries(path, kind, raw_md):
    md = clean_doc_markdown(raw_md)
    page_title = (re.search(r"(?m)^title:\s*(.+)$", raw_md) or [None, path])[1].strip("\"' ")
    sections = parse_sections(md)
    level = entry_level(sections, kind)
    page_url = DOCS_SITE + path.removesuffix(".md").removesuffix("/_index") + "/"

    entries, i = [], 0
    while i < len(sections):
        lvl, title, body = sections[i]
        if lvl != level:
            i += 1
            continue
        subs, intro = [], "\n".join(body)
        j = i + 1
        while j < len(sections) and sections[j][0] > level:
            subs.append((sections[j][1], "\n".join(sections[j][2])))
            j += 1
        problem_extra, resolution = split_problem_and_resolution([(None, intro)] + subs)
        if len(resolution) >= 40 or len(problem_extra) >= 40:
            entries.append((title, problem_extra, resolution or problem_extra))
        i = j
    return page_title, page_url, entries


docs_rows = []
for path, kind in DOC_PAGES:
    resp = requests.get(DOCS_RAW + path, timeout=30)
    if resp.status_code != 200:
        print("skip", path, resp.status_code)
        continue
    page_title, page_url, entries = doc_entries(path, kind, resp.text)
    for n, (title, problem_extra, resolution) in enumerate(entries):
        docs_rows.append({
            "source_id": f"docs:{path}#{n}",
            "source": "docker/docs",
            "source_type": "official_docs_entry",
            "url": page_url,
            "license": DOCS_LICENSE,
            "created_at": None,
            "title": title,
            "problem": problem_extra or title,
            "resolution": resolution,
            "resolution_kind": "official_answer",
            "problem_score": None,
            "resolution_score": None,
            "tags": [kind, page_title],
            "extra": {"page": path, "overlaps_kb": True},
        })

write_jsonl(RAW_SRC / "docs_faq" / "docker_docs_faq.jsonl", docs_rows)
print("docs entries written:", len(docs_rows))

## Part B · Docker Community Forums — solved topics

Discourse serves JSON for any public page. Step 1 lists every topic in each category (30 per request) and keeps those marked solved;
step 2 downloads each kept topic and takes the first post as the problem and the accepted post as the resolution.
Both steps cache to Drive, so a disconnected session resumes where it stopped.

In [ ]:
FORUM = "https://forums.docker.com"
forum_session = requests.Session()
forum_session.headers["User-Agent"] = "docker-triage-research/0.1 (academic project; polite 1 req/s)"
FORUM_DIR = RAW_SRC / "forum"


def forum_get(path, **params):
    for attempt in range(5):
        time.sleep(REQUEST_DELAY)
        resp = forum_session.get(FORUM + path, params=params, timeout=60)
        if resp.status_code == 429:
            time.sleep(30 * (attempt + 1))
            continue
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        return resp.json()
    raise RuntimeError(f"forum kept rate-limiting {path}")


def build_forum_index():
    index_path = FORUM_DIR / "solved_index.jsonl"
    state_path = FORUM_DIR / "index_state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    with index_path.open("a", encoding="utf-8") as out:
        for slug, cat_id in FORUM_CATEGORIES.items():
            page = state.get(slug, 0)
            while page < FORUM_MAX_INDEX_PAGES:
                data = forum_get(f"/c/{slug}/{cat_id}.json", page=page)
                topics = (data or {}).get("topic_list", {}).get("topics", [])
                if not topics:
                    state[slug] = 10 ** 6            # finished this category
                    break
                for t in topics:
                    if t.get("has_accepted_answer") and FORUM_MIN_POSTS <= t["posts_count"] <= FORUM_MAX_POSTS:
                        out.write(json.dumps({"id": t["id"], "slug": t["slug"], "title": t["title"],
                                              "category": slug, "created_at": t["created_at"],
                                              "posts_count": t["posts_count"], "like_count": t.get("like_count", 0)}) + "\n")
                page += 1
                state[slug] = page
                state_path.write_text(json.dumps(state))
                if page % 20 == 0:
                    print(f"  {slug}: listed {page} pages")
    return [json.loads(l) for l in index_path.read_text(encoding="utf-8").splitlines()]


forum_index = build_forum_index()
print("solved topics indexed:", len(forum_index))

In [ ]:
def find_accepted_post(topic_id, topic_json):
    posts = topic_json["post_stream"]["posts"]
    accepted = next((p for p in posts if p.get("accepted_answer")), None)
    missing = [i for i in topic_json["post_stream"]["stream"] if i not in {p["id"] for p in posts}]
    while accepted is None and missing:                      # long thread: fetch the rest in chunks of 20
        chunk, missing = missing[:20], missing[20:]
        data = forum_get(f"/t/{topic_id}/posts.json", post_ids=chunk, include_raw=1)
        more = (data or {}).get("post_stream", {}).get("posts", [])
        accepted = next((p for p in more if p.get("accepted_answer")), None)
    return accepted


def fetch_forum_topics(index):
    out_path = FORUM_DIR / "docker_forum.jsonl"
    done = set()
    if out_path.exists():
        done = {json.loads(l)["source_id"] for l in out_path.read_text(encoding="utf-8").splitlines()}
    written = len(done)
    with out_path.open("a", encoding="utf-8") as out:
        for item in index:
            if written >= FORUM_MAX_TOPICS:
                break
            source_id = f"forum:{item['id']}"
            if source_id in done:
                continue
            topic = forum_get(f"/t/{item['id']}.json", include_raw=1)
            if not topic:
                continue
            question = topic["post_stream"]["posts"][0]
            answer = find_accepted_post(item["id"], topic)
            if not answer or not question.get("raw") or not answer.get("raw"):
                continue
            row = {
                "source_id": source_id,
                "source": "forums.docker.com",
                "source_type": "forum_solved",
                "url": f"{FORUM}/t/{item['slug']}/{item['id']}",
                "license": "Docker forum terms (unspecified; cite by URL)",
                "created_at": item["created_at"],
                "title": item["title"],
                "problem": question["raw"],
                "resolution": answer["raw"],
                "resolution_kind": "forum_accepted_answer",
                "problem_score": item["like_count"],
                "resolution_score": answer.get("like_count", 0),
                "tags": [t["name"] if isinstance(t, dict) else t for t in topic.get("tags", [])] + [item["category"]],
                "extra": {"posts_count": item["posts_count"],
                          "staff_answer": bool(answer.get("staff") or answer.get("moderator") or answer.get("admin")),
                          "answer_trust_level": answer.get("trust_level")},
            }
            out.write(json.dumps(row, ensure_ascii=False) + "\n")
            written += 1
            if written % 50 == 0:
                print(f"  {written} topics collected")
    return written


print("forum rows:", fetch_forum_topics(forum_index))

## Part C · Reddit (optional)

Disabled unless `ENABLE_REDDIT = True` and both secrets exist. A post counts as a ticket when it looks like a question or
problem; its resolution is the highest-scored top-level comment (score ≥ 3), preferring one the original poster replied
to with thanks. There is no accepted-answer marker on Reddit, so these rows get the lowest trust score.

In [ ]:
THANKS_RE = re.compile(r"(?i)\b(thanks|thank you|that worked|worked for me|fixed it|solved|this did it)\b")
QUESTIONISH_RE = re.compile(r"(?i)(\?|error|fail|cannot|can't|not working|issue|problem|help|why)")


def collect_reddit():
    client_id, client_secret = get_secret("REDDIT_CLIENT_ID"), get_secret("REDDIT_CLIENT_SECRET")
    if not (ENABLE_REDDIT and client_id and client_secret):
        print("Reddit skipped (ENABLE_REDDIT is False or credentials missing).")
        return 0
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praw"], check=True)
    import praw

    reddit = praw.Reddit(client_id=client_id, client_secret=client_secret,
                         user_agent="docker-triage-research/0.1 (academic project)")
    rows, seen = [], set()
    for name in REDDIT_SUBREDDITS:
        sub = reddit.subreddit(name)
        listings = [sub.top(time_filter="all", limit=None), sub.top(time_filter="year", limit=None), sub.new(limit=None)]
        for listing in listings:
            for post in listing:
                if len(seen) >= REDDIT_MAX_POSTS:
                    break
                if post.id in seen or post.over_18 or len(post.selftext) < 40 or not QUESTIONISH_RE.search(post.title):
                    continue
                seen.add(post.id)
                post.comment_sort = "top"
                post.comments.replace_more(limit=0)
                top_level = [c for c in post.comments if c.score >= 3 and len(c.body) >= 40 and not c.is_submitter]
                if not top_level:
                    continue
                confirmed = [c for c in top_level if any(r.is_submitter and THANKS_RE.search(r.body) for r in c.replies)]
                best = max(confirmed or top_level, key=lambda c: c.score)
                rows.append({
                    "source_id": f"reddit:{post.id}",
                    "source": f"reddit/r/{name}",
                    "source_type": "reddit_top_comment",
                    "url": "https://www.reddit.com" + post.permalink,
                    "license": "Reddit user content (API terms apply)",
                    "created_at": datetime.fromtimestamp(post.created_utc, tz=timezone.utc).isoformat(),
                    "title": post.title,
                    "problem": post.selftext,
                    "resolution": best.body,
                    "resolution_kind": "op_confirmed_comment" if confirmed else "top_comment",
                    "problem_score": post.score,
                    "resolution_score": best.score,
                    "tags": [post.link_flair_text] if post.link_flair_text else [],
                    "extra": {"num_comments": post.num_comments},
                })
    write_jsonl(RAW_SRC / "reddit" / "reddit.jsonl", rows)
    return len(rows)


print("reddit rows:", collect_reddit())

## Spot-check

In [ ]:
import random

for label, path in [("docs FAQ", RAW_SRC / "docs_faq" / "docker_docs_faq.jsonl"),
                    ("forum", FORUM_DIR / "docker_forum.jsonl"),
                    ("reddit", RAW_SRC / "reddit" / "reddit.jsonl")]:
    if not path.exists():
        continue
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
    print(f"\n=== {label}: {len(rows)} rows")
    for row in random.sample(rows, min(2, len(rows))):
        print(f"* {row['title'][:110]}")
        print("    ->", row["resolution"][:170].replace("\n", " "))